# 04. 네 가지 실패 모드

네 이름은 잘못된 정보, 무관한 정보, 비슷한 대상, 서로 충돌하는 규정이 입력에 들어간 문제를 살펴볼 때 씁니다. 한 사례에 여러 문제가 함께 나타날 수 있습니다. 문제 문장을 넣은 사실과 모델 답에서 실제 오류를 확인한 사실은 따로 기록합니다.

## 실행 환경 준비

실습 파일은 `labs/part2/ch08-context-engineering/notebooks/04-네-가지-실패-모드.ipynb`입니다. 새 커널에서 시작해 첫 코드 셀로 패키지 버전을 맞춘 뒤, 설명과 코드를 위에서부터 순서대로 실행합니다. `.env`는 현재 작업 폴더의 상위 경로에서도 찾습니다.

In [1]:
# %pip은 현재 노트북 커널에 패키지를 설치하는 Jupyter 전용 명령입니다.
# -q는 설치 로그를 줄이고, == 뒤 숫자는 수업에서 검증한 버전을 고정합니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "python-dotenv==1.2.3" "tiktoken==0.14.0"

Note: you may need to restart the kernel to use updated packages.


## 1. API 연결 준비

In [2]:
# find_dotenv는 현재 폴더부터 상위 폴더를 검색해 .env 경로를 찾습니다.
import os

from dotenv import find_dotenv, load_dotenv

# usecwd=True는 노트북을 실행한 폴더를 검색 시작점으로 지정합니다.
# 키 값은 출력하지 않고 존재 여부만 확인합니다.
env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("현재 폴더의 상위 경로에서 .env를 찾지 못했습니다.")
load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env에 OPENAI_API_KEY가 없습니다.")
print(".env와 API 키 확인 완료")

.env와 API 키 확인 완료


### 모델 객체를 만듭니다

`.env` 확인이 끝난 뒤 모델 이름과 출력 제한을 지정합니다. 이 셀은 아직 API에 질문을 보내지 않습니다.

In [3]:
# init_chat_model은 provider:model 형식으로 채팅 모델 객체를 만듭니다.
# model은 이후 셀에서 invoke()를 호출할 때 사용합니다.
from langchain.chat_models import init_chat_model

# openai:는 공급자 이름이며, reasoning_effort="none"은 별도 추론 토큰을 사용하지 않는 설정입니다.
# max_tokens=300은 생성할 출력 토큰 수의 상한입니다.
model = init_chat_model(
    "openai:gpt-5.4-mini",
    reasoning_effort="none",
    max_tokens=300,
)
print("LangChain 모델 준비 완료")

LangChain 모델 준비 완료


## 2. 다섯 가지 입력 조건을 만듭니다

기준 입력을 유지하면서 잘못된 메모, 무관한 정보, 비슷한 기관, 상충 규정을 하나씩 추가합니다.

In [4]:
# clean은 비교의 기준이 되는 근거입니다.
# noise는 길이만 늘리는 무관한 문장입니다.
clean = "[E01 p.2] 미르시립도서관의 정기 휴관일은 오직 매주 월요일 하루뿐입니다."
noise = "회의 안건: 행사 배너와 좌석 배치를 논의했습니다. " * 12
print("기준 문자 수:", len(clean))
print("무관한 문장 문자 수:", len(noise))

기준 문자 수: 44
무관한 문장 문자 수: 348


### 기준 입력에 문제 문장을 하나씩 추가합니다

각 조건은 정상 입력인 `clean`을 그대로 두고, 잘못된 메모·무관한 문장·비슷한 기관·상충 규정 중 하나만 더합니다. 조건 이름과 실제로 추가한 문장을 함께 읽습니다.

In [5]:
# conditions는 기준 입력과 네 실패 자극을 이름별로 보관합니다.
conditions = {
    "clean": clean,
    "poisoning": f"""{clean}
[미검증 메모] 휴관일은 화요일입니다.""",
    "distraction": f"""{noise}
{clean}""",
    "confusion": f"""{clean}
[미르고등학교 도서실] 화요일 휴관입니다.""",
    "clash": f"""{clean}
[합성 부록] 미르시립도서관의 정기 휴관일은 오직 매주 화요일 하루뿐입니다.""",
}
print("실험 조건:", list(conditions))

실험 조건: ['clean', 'poisoning', 'distraction', 'confusion', 'clash']


## 3. 같은 질문으로 다섯 조건을 실행합니다

어떤 실패가 나타날지 미리 정하지 않습니다. 조건 이름, 실제 입력 토큰, 응답 원문을 나란히 기록합니다.

`conditions.items()`는 조건 이름과 입력 내용을 쌍으로 제공합니다. `results[name] = response`는 해당 이름에 실제 응답 객체를 저장합니다. 이름은 개발자가 붙인 실험 조건이며 모델 오류를 자동 진단한 값이 아닙니다.

일반 배송 3일과 도서 지역 배송 5일은 적용 조건이 다릅니다. 같은 조건에 서로 다른 값이 있을 때 충돌을 검토합니다. 이처럼 입력에서 의심할 문제와 실제 출력에서 확인한 오류를 나누어 기록합니다.

In [6]:
# items()는 조건 이름과 근거를 한 쌍씩 반환하며 모든 조건에 같은 질문을 적용합니다.
# results에는 조건별 AIMessage를 저장하고, usage에서 실제 입력 토큰 수를 읽습니다.
results = {}
for name, context in conditions.items():
    messages = [
        ("system", "근거만 사용하고, 충돌하면 확인이 필요하다고 답하세요."),
        ("user", f"""근거:
{context}

질문: 미르시립도서관의 정기 휴관일은 언제인가요?"""),
    ]
    response = model.invoke(messages)
    usage = response.usage_metadata
    if usage is None:
        raise RuntimeError("모델 사용량 메타데이터가 없습니다.")
    results[name] = response
    print(f"\n[{name}] input_tokens={usage['input_tokens']}")
    print(response.text)


[clean] input_tokens=81
미르시립도서관의 정기 휴관일은 **매주 월요일**입니다.



[poisoning] input_tokens=96
근거상 미르시립도서관의 정기 휴관일은 **매주 월요일**입니다.  
다만 미검증 메모에는 **화요일**이라고 되어 있어 충돌하므로, **확인이 필요합니다**.



[distraction] input_tokens=285
미르시립도서관의 정기 휴관일은 **매주 월요일**입니다.



[confusion] input_tokens=96
근거에 따르면 **미르시립도서관의 정기 휴관일은 매주 월요일 하루뿐입니다**.



[clash] input_tokens=111
확인이 필요합니다.  
제공된 근거가 서로 충돌합니다: 한 근거는 **매주 월요일**, 다른 근거는 **매주 화요일**이라고 합니다.


## 4. 출력에서 관찰한 문자열을 비교합니다

요일과 추가 확인 요청이 답에 있는지만 표시합니다. 이 표는 문자열의 포함 여부만 보여 주므로, 답 전체가 맞는지는 응답 원문을 함께 읽어 판단합니다.

In [7]:
# 각 응답에서 요일 문자열과 추가 확인 표현을 차례로 검사합니다.
for name, response in results.items():
    answer = response.text
    observed = []
    for day in ("월요일", "화요일", "수요일"):
        if day in answer:
            observed.append(day)

    if "확인" in answer or "충돌" in answer or "상충" in answer:
        observed.append("추가 확인 표현")

    if not observed:
        observed = ["요일 표현 없음"]
    print(name, observed)

clean ['월요일']
poisoning ['월요일', '화요일', '추가 확인 표현']
distraction ['월요일']
confusion ['월요일']
clash ['월요일', '화요일', '추가 확인 표현']


## 5. 무관 문장 반복 횟수 한 값만 바꿉니다

근거와 질문은 유지하고 반복 횟수만 `12`에서 `24`로 늘립니다. 이 셀은 추가 API 호출 없이 입력 길이 변화만 비교합니다.

In [8]:
# changed_noise는 같은 문장을 24번 연결한 문자열입니다.
changed_noise = "회의 안건: 행사 배너와 좌석 배치를 논의했습니다. " * 24
base_context = noise + clean
changed_context = changed_noise + clean
print("기본 문자 수:", len(base_context))
print("변경 문자 수:", len(changed_context))

기본 문자 수: 392
변경 문자 수: 740


## 정리

실패가 나도록 문제 문장을 넣었다고 해서 모델이 실제로 틀렸다고 기록하지 않습니다. 모델 답에서 확인한 문장을 어떤 입력과 함께 보냈는지 연결해 기록합니다.

## 실패 재현: 상충 규정의 시행일을 기록하지 않습니다

월요일 하루만 휴관한다는 규정과 화요일 하루만 휴관한다는 규정을 함께 전달합니다. 질문 날짜는 같지만 규정의 시행일이 없어 어느 쪽을 적용할지 알 수 없습니다.

In [9]:
# 비교할 날짜와 질문을 먼저 고정합니다.
as_of = "2026-09-08"
question = f"{as_of} 기준 미르시립도서관의 정기 휴관일은 언제인가요?"
# 두 호출 모두 같은 규칙을 사용합니다. 모르는 날짜를 추측하지 않게 합니다.
answer_rule = "근거만 사용하고 출처를 표시하세요. 규정이 충돌하고 시행일을 알 수 없으면 확인이 필요하다고 답하세요."
before_context = conditions["clash"]
print(before_context)

[E01 p.2] 미르시립도서관의 정기 휴관일은 오직 매주 월요일 하루뿐입니다.
[합성 부록] 미르시립도서관의 정기 휴관일은 오직 매주 화요일 하루뿐입니다.


### 같은 질문으로 실제 답을 읽습니다

임의의 오류를 내는 검사 대신 모델이 두 규정을 어떻게 설명하는지 확인합니다. 확인이 필요하다고 답했다면 잘못된 규정을 고른 실패는 관찰되지 않은 것입니다. 다만 현재 자료만으로 질문에 확정 답변을 할 수는 없습니다.

In [10]:
# 두 규정의 본문과 질문을 그대로 전달합니다.
before_prompt = f"""근거:
{before_context}

질문: {question}"""
before_response = model.invoke([
    ("system", answer_rule),
    ("user", before_prompt),
])
print(before_response.text)

근거가 서로 충돌합니다.

- [E01 p.2]에는 **정기 휴관일이 매주 월요일**이라고 되어 있습니다.
- [합성 부록]에는 **정기 휴관일이 매주 화요일**이라고 되어 있습니다.

따라서 **2026-09-08 기준 미르시립도서관의 정기 휴관일이 언제인지 확정할 수 없으며, 확인이 필요합니다.**


## 수정: 확인한 시행일로 적용할 규정을 고릅니다

이 연습에서는 기관에 확인한 결과 기존 규정의 시행일은 2025-01-01, 대체 규정의 시행일은 2026-01-01이라고 가정합니다. 날짜는 실습용으로 정한 값이며 실제 기관의 규정이 아닙니다. 두 규정의 본문은 그대로 두고, 같은 업무의 새 규정이 이전 규정을 대체한다는 조건에서 기준일까지 시행된 가장 최근 규정을 선택합니다.

`eligible`은 기준일 전에 시행된 후보 목록입니다. `max(eligible, key=effective_date)`는 각 후보를 `effective_date()`에 넘겨 얻은 날짜 문자열을 비교합니다. 함수 이름을 넘기므로 비교할 때마다 필요한 후보의 날짜를 읽습니다. 후보가 비어 있거나 서로 다른 날짜 형식이 섞인 경우는 이 예제의 입력에 없으며 실제 처리에서는 별도 분기가 필요합니다.

In [11]:
# versioned는 같은 두 규정에 시행일만 추가한 목록이고 as_of는 질문한 날짜입니다.
# eligible에서 기준일까지 시행된 규정만 남긴 뒤 max의 key로 가장 최근 시행일을 고릅니다.
# YYYY-MM-DD처럼 자릿수가 고정된 날짜는 문자열 순서와 날짜 순서가 같습니다. 다른 형식은 날짜 객체로 바꿉니다.
versioned = [
    {"effective_date": "2025-01-01", "rule": clean, "source": "E01 p.2"},
    {"effective_date": "2026-01-01", "rule": conditions["clash"].splitlines()[-1],
     "source": "합성 부록"},
]
eligible = []
for record in versioned:
    if record["effective_date"] <= as_of:
        eligible.append(record)

def effective_date(record: dict[str, str]) -> str:
    return record["effective_date"]

selected_rule = max(eligible, key=effective_date)
assert "오직 매주 화요일 하루뿐" in selected_rule["rule"]
assert selected_rule["source"] == "합성 부록"
print("수정 후 적용 규정:", selected_rule)

수정 후 적용 규정: {'effective_date': '2026-01-01', 'rule': '[합성 부록] 미르시립도서관의 정기 휴관일은 오직 매주 화요일 하루뿐입니다.', 'source': '합성 부록'}


### 선택한 규정으로 같은 질문에 다시 답합니다

새 규정을 고른 결과를 근거로 넣습니다. 질문·모델·답변 규칙은 Before와 같습니다. 실제 답이 화요일과 선택한 출처를 사용하는지 확인합니다. 이는 시행일을 확인하고 선택한 효과이며, 모델 자체의 성능이 높아졌다는 뜻은 아닙니다.

In [12]:
# 날짜로 선택한 규정만 모델에 전달합니다.
after_prompt = f"""근거:
{selected_rule['rule']}
시행일: {selected_rule['effective_date']}

질문: {question}"""
after_response = model.invoke([
    ("system", answer_rule),
    ("user", after_prompt),
])
print(after_response.text)

2026-09-08 기준으로 **미르시립도서관의 정기 휴관일은 매주 화요일**입니다.  
근거: “[합성 부록] 미르시립도서관의 정기 휴관일은 오직 매주 화요일 하루뿐입니다.” (시행일: 2026-01-01)

따라서 **2026-09-08은 화요일이므로 정기 휴관일**입니다.
